### 3.9.47. Discrete-Time Optimal Control Problems

$$
\begin{aligned}
\min_{u_0,\ldots,u_{N-1}}\quad
& G_N(x_N)+\sum_{k=0}^{N-1}G_k(x_k,u_k) \\
\text{subject to}\quad
& x_{k+1}=F_k(x_k,u_k),\quad k=0,\ldots,N-1.
\end{aligned}
$$


**Explanation:**

Discrete-time optimal control fits naturally into unconstrained optimization because, after the state recursion is eliminated, the control sequence becomes the decision vector of a large unconstrained optimization problem. The dynamics still matter because each control affects later states, but the reduced cost can be viewed as a function of $u_0,\ldots,u_{N-1}$. Gradients can be computed by forward sensitivity equations or by backward adjoint recursions, both of which exploit the time structure rather than treating the problem as an arbitrary dense optimization problem.

The decision variables are the controls $u_0,\ldots,u_{N-1}$. The objective adds terminal cost $G_N(x_N)$ and stage costs $G_k(x_k,u_k)$, while the constraints say each state is produced from the previous state and control by the dynamics $F_k$.

This section is a bridge to optimal control. It shows why the algorithms from unconstrained optimization are not isolated tools: gradient, conjugate-gradient, and Newton-like methods can all be applied to control problems once the cost is expressed in terms of the control sequence. The large dimension and special structure strongly influence which method is practical.

**Intuition:**

In a finite-horizon scalar system, a proposed control sequence determines every state by forward simulation. The objective can then be evaluated from the resulting state and control histories. The code cell carries out this reduction for a short horizon, turning a dynamic problem into a scalar cost associated with one control vector.


**Numerical Example:**

Use the scalar dynamics
$$
x_{k+1}=1.1x_k+u_k,
\qquad x_0=2,
\qquad (u_0,u_1,u_2)=(-0.8,-0.5,-0.2).
$$

Forward simulation gives
$$
\begin{aligned}
x_1&=1.1(2)-0.8=1.4,\\
x_2&=1.1(1.4)-0.5=1.04,\\
x_3&=1.1(1.04)-0.2=0.944.
\end{aligned}
$$

With running cost $x_k^2+0.1u_k^2$ and terminal cost $x_3^2$,
$$
\begin{aligned}
G_0&=2^2+0.1(-0.8)^2=4+0.064=4.064,\\
G_1&=1.4^2+0.1(-0.5)^2=1.96+0.025=1.985,\\
G_2&=1.04^2+0.1(-0.2)^2=1.0816+0.004=1.0856,\\
G_3&=0.944^2=0.891136.
\end{aligned}
$$

The total cost for this control sequence is
$$
4.064+1.985+1.0856+0.891136=8.025736.
$$


In [1]:
import sympy as sp

state_coefficient = sp.Rational(11, 10)
control_coefficient = sp.Integer(1)
initial_state = sp.Integer(2)
control_sequence = [sp.Rational(-4, 5), sp.Rational(-1, 2), sp.Rational(-1, 5)]

states = [initial_state]
running_cost = sp.Integer(0)
for control_value in control_sequence:
    running_cost += states[-1] ** 2 + sp.Rational(1, 10) * control_value ** 2
    states.append(state_coefficient * states[-1] + control_coefficient * control_value)
terminal_cost = states[-1] ** 2
total_cost = running_cost + terminal_cost

print("states =", states)
print("total_cost =", total_cost, "=", float(total_cost))

states = [2, 7/5, 26/25, 118/125]
total_cost = 1003217/125000 = 8.025736


**Equivalent `casadi` implementation:**

In [2]:
import casadi as ca

state = ca.SX.sym("x")
control = ca.SX.sym("u")
state_coefficient = 1.1
control_coefficient = 1.0
dynamics = ca.Function("f", [state, control], [state_coefficient * state + control_coefficient * control])
stage_cost = ca.Function("l", [state, control], [state ** 2 + 0.1 * control ** 2])

initial_state = 2.0
control_sequence = [-0.8, -0.5, -0.2]
states = [ca.DM(initial_state)]
running_cost = ca.DM(0)
for control_value in control_sequence:
    running_cost = running_cost + stage_cost(states[-1], control_value)
    states.append(dynamics(states[-1], control_value))
terminal_cost = states[-1] ** 2
total_cost = running_cost + terminal_cost

print("states =", [float(state_value) for state_value in states])
print("total_cost =", total_cost)

states = [2.0, 1.4000000000000001, 1.0400000000000003, 0.9440000000000004]
total_cost = 8.02574


**References:**

[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.), Chapter 1 "Unconstrained Optimization". Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Interior-Point Implementation Patterns](./46_interior_point_implementation_patterns.ipynb) | [Next: Practical Guidelines ➡️](./48_practical_guidelines.ipynb)

---
